# The one-sample t-test
The chocolate-bar test from summary numbers, a t-test on Titanic ages, and how often a sample of 25 rejects H0.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## 1. Chocolate bars: only summary statistics

In [ ]:
x_bar, mu0, s, n = 49.7, 50, 1.2, 25
t = (x_bar - mu0) / (s / np.sqrt(n))
# two-tailed p-value from the t CDF with n - 1 degrees of freedom
p = 2 * stats.t.cdf(-abs(t), df=n - 1)
print(f"t = {t:.2f}, df = {n - 1}, p = {p:.3f}")
# the same numbers on the normal curve, for comparison
print(f"z-test p would be {2 * stats.norm.cdf(-abs(t)):.3f}")

## 2. Titanic ages: the population and one sample
Train and test files together hold 1309 passengers; 1046 have a known age.

In [ ]:
titanic = pd.concat([pd.read_csv("data/titanic_train.csv"), pd.read_csv("data/titanic_test.csv")])
ages = titanic["Age"].dropna().reset_index(drop=True)
print(len(titanic), "passengers,", len(ages), "known ages, true mean", round(ages.mean(), 2))
# one random sample of 25 ages
sample_age = ages.sample(25, random_state=0)
print(f"sample mean {sample_age.mean():.2f}, sample sd {sample_age.std():.2f}")

## 3. Check normality with the Shapiro-Wilk test
H0 of this test: the data comes from a normal distribution.

In [ ]:
print("Shapiro-Wilk p, sample of 25:", round(stats.shapiro(sample_age).pvalue, 3))
print("Shapiro-Wilk p, all 1046 ages:", f"{stats.shapiro(ages).pvalue:.1e}")

## 4. H0: mu = 35 against H1: mu < 35

In [ ]:
res = stats.ttest_1samp(sample_age, popmean=35, alternative="less")
print(f"t = {res.statistic:.2f}, one-tailed p = {res.pvalue:.4f}")
# by hand: same t and p
t_hand = (sample_age.mean() - 35) / (sample_age.std() / np.sqrt(25))
print(f"by hand: t = {t_hand:.2f}, p = {stats.t.cdf(t_hand, df=24):.4f}")
# the default is two-sided
print(f"two-sided p = {stats.ttest_1samp(sample_age, popmean=35).pvalue:.4f}")

## 5. How often does a sample of 25 reject H0? (the power)

In [ ]:
rejections = [stats.ttest_1samp(ages.sample(25, random_state=i), 35, alternative="less").pvalue <= 0.05
              for i in range(4000)]
print("share of samples that reject H0:", np.mean(rejections))

## 6. The five videos: is 9 minutes on average better than 6?

In [ ]:
videos = [7, 9, 5, 11, 13]
res = stats.ttest_1samp(videos, popmean=6, alternative="greater")
print(f"t = {res.statistic:.2f}, p = {res.pvalue:.3f}")